# Poster2Movie — Clasificador del equipo: reproducir la base

Este notebook comprueba que el repo del equipo reproduce el sistema base (CLIP ViT-B/32 congelado + capa lineal + umbral por género) y publica el release `v0.1.0`.

**Entorno:** CPU basta (entrenar la cabeza tarda segundos; los vectores de la base ya están calculados en el dataset compartido).

## Preparar el entorno

- `P2M_DATOS`: el dataset compartido (solo lectura). Agrégalo a *Mi unidad* como acceso directo.
- `P2M_SALIDAS`: la carpeta del equipo en Drive, donde se guardan vectores, modelos y predicciones.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"
os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_clasificador"
REPO = "11909/Poster2Movie-Clasificador"
if not os.path.isdir("/content/clasificador"):
    !git clone --recurse-submodules https://github.com/{REPO}.git /content/clasificador
else:
    !cd /content/clasificador && git pull && git submodule update --init
%cd /content/clasificador
!pip install -q -r requirements.txt

## 1. Entrenar la base

Usa los vectores `clip_b32__centro`, que son los `embeddings_clip_b32.npz` que ya están en el dataset compartido.

In [ ]:
!python -m clasificador.entrenar experimentos/base.yaml

## 2. Evaluar en validación

Esto es lo que se hace con cada experimento. Escribe `resultados/base/` y agrega una fila a `resultados/registro.csv`.

In [ ]:
!python -m clasificador.evaluar base

## 3. Publicar `v0.1.0`

Solo una vez. Mide en validación **y en test**: en test debe dar F1 macro **0.537**, igual que el repo principal. Si no da, algo está mal: no publiquen y avisen.

In [ ]:
!python -m clasificador.release v0.1.0 base --notas "Base del repo principal reproducida en el repo del equipo."

## 4. Subir a GitHub

Necesitas un token de GitHub con permiso de escritura a este repo (*Settings → Developer settings → Fine-grained tokens*). Al subir la etiqueta, GitHub Actions crea el release solo.

In [ ]:
from getpass import getpass
token = getpass("Token de GitHub: ")
!git config user.name "{input('Tu nombre: ')}"
!git config user.email "{input('Tu correo de GitHub: ')}"
!git add releases/v0.1.0 resultados
!git commit -m "Release v0.1.0: base reproducida"
!git tag v0.1.0
!git push https://{token}@github.com/{REPO}.git HEAD:main --tags
del token

## 5. Probar con un cartel

In [ ]:
from google.colab import files
from PIL import Image
from clasificador.predictor import Clasificador
from p2m.config import GENEROS_ES

clasificador = Clasificador()  # el release más reciente
for nombre in files.upload():
    resultado = clasificador.predecir(Image.open(nombre).convert("RGB"))
    print(nombre, "->", ", ".join(GENEROS_ES[g] for g in resultado["generos"]))
    top = sorted(resultado["probabilidades"].items(), key=lambda x: -x[1])[:5]
    print("   ", ", ".join(f"{GENEROS_ES[g]} {p:.2f}" for g, p in top))